In [ ]:
from netsuite_auth import get_auth, run_suiteql

auth, BASE_URL = get_auth()

In [ ]:
# --- Permissions Check ---
# Verifies the configured role has access to every record type listed below.
# Each check runs a minimal query (rownum <= 1) and reports PASS, WARN, or FAIL.
#
# PASS  — query returned at least one row
# WARN  — query returned 0 rows; NetSuite silently returns empty (HTTP 200, 0 rows)
#         when access is denied via SuiteQL rather than raising an error.
#         This usually means the role is missing View permission for that record type.
# FAIL  — query raised an exception (network error, syntax error, etc.)
#
# Add or remove entries from `checks` to match the record types your notebooks use.
# Note: viewing journal entries requires Transactions -> Make Journal Entry -> View,
# not a permission named "Journal Entries".

checks = [
    # (label, query, required_by)
    ("item",
     "SELECT id FROM item WHERE rownum <= 1",
     "item_master_schema"),

    ("transaction — Invoice (CustInvc)",
     "SELECT id FROM transaction WHERE type = 'CustInvc' AND rownum <= 1",
     "invoice queries"),

    ("transaction — Sales Order (SalesOrd)",
     "SELECT id FROM transaction WHERE type = 'SalesOrd' AND rownum <= 1",
     "sales_order_query"),

    ("transaction — Journal Entry (Journal) [requires: Make Journal Entry -> View]",
     "SELECT id FROM transaction WHERE type = 'Journal' AND rownum <= 1",
     "GL analysis"),

    ("transactionline",
     "SELECT id FROM transactionline WHERE rownum <= 1",
     "line-level queries"),

    ("account",
     "SELECT id FROM account WHERE rownum <= 1",
     "account lookups"),

    ("customer",
     "SELECT id FROM customer WHERE rownum <= 1",
     "customer queries"),

    ("vendor",
     "SELECT id FROM vendor WHERE rownum <= 1",
     "vendor queries"),

    ("employee",
     "SELECT id FROM employee WHERE rownum <= 1",
     "employee queries"),
]

results = []
all_passed = True

for label, sql, used_by in checks:
    try:
        rows = run_suiteql(sql, limit=1)
        if rows:
            status = "PASS"
            detail = f"returned {len(rows)} row(s)"
            icon = "✓"
        else:
            status = "WARN"
            detail = "0 rows — may indicate missing View permission"
            icon = "⚠"
            all_passed = False
    except Exception as e:
        status = "FAIL"
        detail = str(e)
        icon = "✗"
        all_passed = False

    results.append({"record_type": label, "status": status, "detail": detail, "required_by": used_by})
    print(f"  {icon}  {label:<60}  {status}")

print()
if all_passed:
    print("All checks passed.")
else:
    print("One or more checks returned WARN or FAIL.")
    print("WARN: verify the role has View permission for that record type in NetSuite.")
    print("FAIL: review the detail column for the error.")

In [ ]:
# --- Detail table ---
import pandas as pd

df = pd.DataFrame(results)
df